# Diffusion Noise Schedules（扩散噪声调度）

**难度：** Easy

**函数签名：** `noise_schedule(num_timesteps, schedule_type='cosine') -> Tensor`

**参数：**
- `num_timesteps` — 扩散时间步数 T
- `schedule_type` — `'linear'`、`'cosine'` 或 `'sigmoid'`

**返回：** 形状 `(num_timesteps,)` 的 alpha_bar 张量，值域 (0, 1]，单调递减

**公式：**
- `linear`: β_t 线性插值，ᾱ = cumprod(1-β)
- `cosine`: ᾱ_t = cos²(...) / cos²(...)
- `sigmoid`: β_t 经 sigmoid 变换，ᾱ = cumprod(1-β)

In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def noise_schedule(num_timesteps, schedule_type='cosine'):
    T = num_timesteps
    # t = [0, 1, 2, ..., T-1]，用作时间索引
    t = torch.arange(T, dtype=torch.float32)

    if schedule_type == 'linear':
        # ---- Linear 调度 ----
        # β 从 β_start 线性增长到 β_end
        # β 控制每步添加多少噪声
        beta_start, beta_end = 1e-4, 0.02
        # 线性插值: β_t = β_start + (β_end - β_start) * t / (T-1)
        # t=0 → β_start, t=T-1 → β_end
        betas = beta_start + (beta_end - beta_start) * t / (T - 1)
        # ᾱ_t = ∏_{s=0}^{t} (1 - β_s) = cumprod(1-β)
        # ᾱ_t 是从开始到 t 步的累积"保留信号"比例
        # ᾱ_0 ≈ 1 (几乎纯信号), ᾱ_{T-1} ≈ 0 (几乎纯噪声)
        alpha_bar = torch.cumprod(1 - betas, dim=0)

    elif schedule_type == 'cosine':
        # ---- Cosine 调度 ----
        # 来自 "Improved DDPM" 论文
        # 直觉: cos 函数在 [0, π/2] 上从 1 平滑降到 0
        # s=0.008 是偏移量，防止 t=0 时 ᾱ 精确为 1
        s = 0.008
        # f(t) = cos²((t/T + s)/(1+s) * π/2)
        # t=0: f(0) = cos²(s/(1+s) * π/2) ≈ cos²(0.0125) ≈ 0.9997
        # t=T: f(T) = cos²((1+s)/(1+s) * π/2) = cos²(π/2) = 0
        f = torch.cos(((t / T + s) / (1 + s)) * (math.pi / 2)) ** 2
        # 归一化: ᾱ = f(t) / f(0)，确保 ᾱ_0 = 1
        f0 = math.cos((s / (1 + s)) * (math.pi / 2)) ** 2
        # clamp 防止极端值: ᾱ 不能精确为 0 或 1
        alpha_bar = (f / f0).clamp(0.0001, 0.9999)

    elif schedule_type == 'sigmoid':
        # ---- Sigmoid 调度 ----
        # 用 sigmoid 函数生成平滑的 β 调度
        # sigmoid(x) = 1/(1+exp(-x))，S 形曲线
        beta_start, beta_end = 1e-4, 0.02
        # x = t/T * 12 - 6: 将 [0, T] 映射到 [-6, 6]
        # sigmoid(-6) ≈ 0.0025, sigmoid(6) ≈ 0.9975
        # 这样 sigmoid 的有效变化区间大致在中间
        x = t / T * 12 - 6
        sig = 1 / (1 + torch.exp(-x))
        # 缩放 sigmoid 到 [β_start, β_end] 范围
        # (sig - min)/(max - min) 归一化到 [0, 1]
        # 再线性映射到 [β_start, β_end]
        betas = beta_start + (beta_end - beta_start) * (sig - sig.min()) / (sig.max() - sig.min())
        alpha_bar = torch.cumprod(1 - betas, dim=0)

    else:
        raise ValueError(f'Unknown schedule_type: {schedule_type}')

    return alpha_bar

In [ ]:
T = 1000
schedules = ['linear', 'cosine', 'sigmoid']
checkpoints = [0, T // 4, T // 2, 3 * T // 4, T - 1]
labels = ['t=0', 'T/4', 'T/2', '3T/4', 'T-1']

# Header
col_w = 10
header = f"{'t':>6}" + "".join(f"{s:>{col_w}}" for s in schedules)
print(header)
print("-" * len(header))

results = {s: noise_schedule(T, s) for s in schedules}
for label, idx in zip(labels, checkpoints):
    row = f"{label:>6}" + "".join(f"{results[s][idx].item():>{col_w}.4f}" for s in schedules)
    print(row)

In [ ]:
from torch_judge import check
check("noise_schedule")

## 📝 核心思路总结

1. **cumprod 是关键**：`ᾱ = cumprod(1-β)`，累积乘积计算信号保留比例
2. **三种调度各有特点**：linear 简单，cosine 平滑，sigmoid S 形
3. **ᾱ 的含义**：ᾱ_t 是 t 步后剩余信号的比例，ᾱ_0≈1，ᾱ_{T-1}≈0
4. **clamp 防极端**：ᾱ 不能精确为 0 或 1，避免数值问题